# Gamma Operator: Ancilla vs Ancilla-Free Verification

This notebook implements two constructions of the **Gamma diagonal phase operator** from the Jiang et al. 2D FFFT paper (arXiv:1711.05395), then verifies they produce identical unitaries.

**Gamma** is diagonal in the computational basis: $\Gamma|s\rangle = (-1)^{f(s)}|s\rangle$ where $f(s)$ is a degree-2 polynomial over GF(2). It satisfies the key property $(\star)$: for any vertical hop between snake-order sites $j < k$, and basis states $|s\rangle, |s'\rangle$ differing only at $j,k$:

$$\gamma_s \cdot \gamma_{s'} = (-1)^{\sum_{l=j+1}^{k-1} s_l}$$

**Construction 1** (Jiang et al.): Uses $L$ ancilla qubits (one per row). Four stages: column parity cascade, parity-basis CZ sweep, inverse cascade, original-basis CZ sweep.

**Construction 2** (Ancilla-free): Same operator via $\Gamma = D_D \cdot C^{-1} \cdot D_B \cdot C$ using row CNOT cascades + nearest-neighbor CZ. Zero ancillas, $O(\sqrt{N})$ depth, $O(N)$ gates.

In [1]:
import cirq
import numpy as np
from typing import Dict, List, Tuple, Optional, Sequence

# --- Snake ordering utilities ---

def snake_order_indices(L: int) -> List[int]:
    """Return raster indices in snake order.
    Even rows (0,2,...) go L->R, odd rows go R->L."""
    order = []
    for r in range(L):
        row = [r * L + c for c in range(L)]
        if r % 2 == 1:
            row.reverse()
        order.extend(row)
    return order


def rc_to_snake(r: int, c: int, L: int) -> int:
    """Convert (row, col) to snake-order site index."""
    if r % 2 == 0:  # L-row
        return r * L + c
    else:  # R-row
        return r * L + (L - 1 - c)


def snake_to_rc(idx: int, L: int) -> Tuple[int, int]:
    """Convert snake-order site index to (row, col)."""
    r = idx // L
    pos_in_row = idx % L
    if r % 2 == 0:
        c = pos_in_row
    else:
        c = L - 1 - pos_in_row
    return r, c


def is_L_row(r: int) -> bool:
    """L-rows are even-indexed (go left->right in snake)."""
    return r % 2 == 0


def sites_between(r1: int, c: int, r2: int, L: int) -> List[int]:
    """Snake-order site indices strictly between (r1,c) and (r2,c).
    Assumes r2 = r1 + 1 (vertical hop on same column)."""
    j = rc_to_snake(r1, c, L)
    k = rc_to_snake(r2, c, L)
    lo, hi = min(j, k), max(j, k)
    return list(range(lo + 1, hi))


# --- Qubit constructors ---

def make_system_qubits(L: int) -> Dict[Tuple[int, int], cirq.GridQubit]:
    return {(r, c): cirq.GridQubit(r, c) for r in range(L) for c in range(L)}


def make_ancilla_qubits(L: int) -> Dict[int, cirq.NamedQubit]:
    return {r: cirq.NamedQubit(f"anc_r{r}") for r in range(L)}


def sys_qubit(sq: Dict, r: int, c: int) -> cirq.GridQubit:
    return sq[(r, c)]


# Quick test
L = 3
print("Snake order for L=3:", snake_order_indices(L))
print("rc_to_snake(1,2,3):", rc_to_snake(1, 2, 3))
print("sites_between(0,0, 1,3):", sites_between(0, 0, 1, 3))

Snake order for L=3: [0, 1, 2, 5, 4, 3, 6, 7, 8]
rc_to_snake(1,2,3): 3
sites_between(0,0, 1,3): [1, 2, 3, 4]


## Shared: Column Parity Cascade

Used by both constructions. Forward (Stage A / Phase 1): CNOT cascade bottom-to-top on each column, so qubit $(r,c)$ becomes $\tilde{s}_{r,c} = \bigoplus_{r' \ge r} s_{r',c}$. Inverse (Stage C / Phase 3): top-to-bottom to undo.

In [2]:
def column_parity_cascade_ops(sq: Dict, L: int, inverse: bool = False) -> List[cirq.Operation]:
    """Column parity CNOT cascade.
    Forward: bottom-to-top, CNOT(r+1,c -> r,c) for r from L-2 down to 0.
    Inverse: top-to-bottom, CNOT(r+1,c -> r,c) for r from 0 to L-2.
    Returns ops grouped by layer (all columns in parallel per layer)."""
    ops = []
    if not inverse:
        for r in range(L - 2, -1, -1):
            for c in range(L):
                ops.append(cirq.CNOT(sq[(r + 1, c)], sq[(r, c)]))
    else:
        for r in range(L - 1):
            for c in range(L):
                ops.append(cirq.CNOT(sq[(r + 1, c)], sq[(r, c)]))
    return ops

## Construction 1: Gamma with Ancillas (Jiang et al.)

One ancilla per row. The ancilla sweeps across columns accumulating parity via CNOTs.

For **simulation purposes**, the ancilla is a persistent `NamedQubit` that receives CNOTs from successive system qubits. The physical SWAPs in the paper serve only nearest-neighbor locality — the logical effect is identical to direct CNOT accumulation. We note this equivalence and skip the SWAPs.

**Stage A**: Column parity cascade (forward)
**Stage B**: Parity-basis CZ sweep (right→left). Order per step $p$: CZ gates, then CNOT to accumulate.
- Skip-row: $\text{CZ}(\text{sys}(r,p), \text{anc}(r+2))$ for L-rows $r$ with $r+2 \le L-1$
- Same-row: $\text{CZ}(\text{sys}(r,p), \text{anc}(r))$ for L-rows $r \ge 2$
- CNOT: $\text{CNOT}(\text{sys}(r,p) \to \text{anc}(r))$ for all rows

**Stage C**: Column parity cascade (inverse), applied to both system and ancilla qubits
**Stage D**: Original-basis CZ sweep (left→right). Order per step $p$: CNOT to peel, then CZ gates.
- CNOT: $\text{CNOT}(\text{sys}(r,p) \to \text{anc}(r))$ for all rows
- CZ: for each L-row $r$: $\text{CZ}(\text{sys}(r,p), \text{anc}(r))$ and $\text{CZ}(\text{sys}(r,p), \text{anc}(r{+}1))$ if $r{+}1 < L$

**Note on odd $L$:** The bottom L-row ($r = L-1$) has no right-closed partner when $L$ is odd.
Without $\text{CZ}(\text{sys}(L{-}1,p), \text{anc}(L{-}1))$ in Stage D, the boundary terms from Stage B
for the bottom left-closed pair fail to cancel. The fix: include a self-CZ for the orphan row.

In [3]:
# =============================================================================
# Construction 1: Gamma with Ancillas (Jiang et al.)
#
# CNOT depth: 7L - 3  (exact, verified empirically for L = 3..21)
# Gate count:  O(N)   where N = L^2
# Ancillas:    L      (one per row)
#
# Per-stage CNOT depth breakdown:
#   Stage A  (col cascade forward):   L - 1
#   Stage B  (R→L CZ sweep):          3L
#     Each of L steps: CZ layer depth 2 + CNOT layer depth 1 = 3.
#     No inter-step overlap (consecutive steps share ancilla qubits).
#   Stage C  (inverse cascades):       L - 1
#     System and ancilla cascades run fully in parallel (disjoint qubits).
#   Stage D  (L→R CZ sweep):           2L + 1
#     Each of L steps: CNOT depth 1 + CZ depth 2 = 3.
#     Inter-step overlap of 1: step p+1's CNOT uses different qubits
#     from step p's second CZ → 3L - (L-1) = 2L + 1.
#   Inter-stage overlap:               -2
#     Stage C's last cascade moment overlaps with Stage D's first CNOT.
#   ─────────────────────────────────────
#   Total: (L-1) + 3L + (L-1) + (2L+1) - 2 = 7L - 3
#
# No further depth reduction is possible within this decomposition:
#   - Stage B: CZ must precede CNOT at each step; consecutive steps share
#     ancilla qubits, so no inter-step overlap. 3L is tight.
#   - Stage D: the single inter-step overlap is already exploited. 2L+1 is tight.
#   - Stages A/C: sequential column cascades, L-1 is minimal.
# =============================================================================

def build_stage_B_ops(sq: Dict, aq: Dict, L: int) -> List[cirq.Operation]:
    """Stage B: parity-basis CZ sweep, ancillas sweep right->left.
    At each step p (from L-1 down to 0):
      1. CZ gates (skip-row and same-row) — depth 2
      2. CNOT(sys(r,p) -> anc(r)) to accumulate parity — depth 1
    CNOT depth per step: 3.  Total: 3L."""
    ops = []
    for p in range(L - 1, -1, -1):
        # CZ gates: skip-row and same-row (depth 2: skip CZs parallel, then same CZs parallel)
        for r in range(0, L, 2):  # L-rows only
            if r + 2 <= L - 1:
                ops.append(cirq.CZ(sq[(r, p)], aq[r + 2]))
            if r >= 2:
                ops.append(cirq.CZ(sq[(r, p)], aq[r]))
        # CNOT to accumulate parity into ancilla (depth 1: all rows parallel)
        for r in range(L):
            ops.append(cirq.CNOT(sq[(r, p)], aq[r]))
    return ops


def ancilla_column_cascade_ops(aq: Dict, L: int, inverse: bool = False) -> List[cirq.Operation]:
    """CNOT cascade on the ancilla 'column'.
    Forward (bottom-to-top): CNOT(anc(r+1) -> anc(r)) for r from L-2 down to 0.
    Inverse (top-to-bottom): CNOT(anc(r+1) -> anc(r)) for r from 0 to L-2.
    CNOT depth: L-1 (sequential, each CNOT depends on previous)."""
    if not inverse:
        return [cirq.CNOT(aq[r + 1], aq[r]) for r in range(L - 2, -1, -1)]
    else:
        return [cirq.CNOT(aq[r + 1], aq[r]) for r in range(L - 1)]


def build_stage_D_ops(sq: Dict, aq: Dict, L: int) -> List[cirq.Operation]:
    """Stage D: original-basis CZ sweep, ancillas sweep left->right.
    At each step p (from 0 to L-1):
      1. CNOT(sys(r,p) -> anc(r)) to peel off column — depth 1
      2. CZ gates for right-closed pairs — depth 2
         (CZ(sys(r,p), anc(r)) and CZ(sys(r,p), anc(r+1)) share sys(r,p))
    CNOT depth per step: 3.  Inter-step overlap: 1.  Total: 2L + 1."""
    ops = []
    for p in range(L):
        # CNOT to peel off column p (depth 1: all rows parallel)
        for r in range(L):
            ops.append(cirq.CNOT(sq[(r, p)], aq[r]))
        # CZ gates for each L-row (depth 2: two CZs share sys(r,p))
        for r in range(0, L, 2):
            if r + 1 < L:
                # Right-closed pair (r, r+1)
                ops.append(cirq.CZ(sq[(r, p)], aq[r]))
                ops.append(cirq.CZ(sq[(r, p)], aq[r + 1]))
            else:
                # Bottom L-row with no partner (odd L only)
                ops.append(cirq.CZ(sq[(r, p)], aq[r]))
    return ops


def build_gamma_with_ancillas(L: int):
    """Build the full Gamma circuit with ancillas (Jiang et al.).
    Returns (circuit, system_qubit_list, ancilla_qubit_list).

    CNOT depth: 7L - 3  (exact for all L >= 3, verified empirically)
    Gate count:  O(N)
    Ancillas:    L"""
    sq = make_system_qubits(L)
    aq = make_ancilla_qubits(L)

    ops = []
    # Stage A: column parity cascade (forward) — depth L-1
    ops.extend(column_parity_cascade_ops(sq, L, inverse=False))
    # Stage B: parity-basis CZ sweep — depth 3L
    ops.extend(build_stage_B_ops(sq, aq, L))
    # Stage C: inverse cascade on system + ancilla column — depth L-1
    # (system and ancilla cascades are on disjoint qubit sets → fully parallel)
    ops.extend(column_parity_cascade_ops(sq, L, inverse=True))
    ops.extend(ancilla_column_cascade_ops(aq, L, inverse=True))
    # Stage D: original-basis CZ sweep — depth 2L+1
    ops.extend(build_stage_D_ops(sq, aq, L))

    circuit = cirq.Circuit(ops)
    sys_list = [sq[(r, c)] for r in range(L) for c in range(L)]
    anc_list = [aq[r] for r in range(L)]
    return circuit, sys_list, anc_list


# Quick check: build for L=3 and verify depth
c1, sys1, anc1 = build_gamma_with_ancillas(3)
expected = 7 * 3 - 3
print(f"Construction 1 (L=3): {len(c1)} moments, {len(sys1)} sys qubits, {len(anc1)} ancillas")
print(f"  Expected CNOT depth: 7*3 - 3 = {expected}, actual = {len(c1)}")
assert len(c1) == expected, f"Depth mismatch: got {len(c1)}, expected {expected}"

Construction 1 (L=3): 18 moments, 9 sys qubits, 3 ancillas
  Expected CNOT depth: 7*3 - 3 = 18, actual = 18


## Construction 2: Ancilla-Free Gamma

Implements $\Gamma = D_D \cdot C^{-1} \cdot D_B \cdot C$ using only system qubits.

The key primitive: $T(x, y) = \bigoplus_{p < c'} x_p \cdot y_{c'}$ implemented via CNOT cascade + CZ + uncascade.

**Phase 1** ($C$): Column parity cascade forward

**Phase 2** ($D_B$): Parity-basis CZ interactions
- 2a: Same-row for L-rows $r \ge 2$: suffix cascade → adjacent CZ → uncascade → Z on odd columns
- 2b: Skip-row for L-rows $r$ with $r+2 \le L-1$: prefix cascade on row $r$, route CZ through row $r+1$ to reach $r+2$

**Phase 3** ($C^{-1}$): Column parity cascade inverse

**Phase 4** ($D_D$): Original-basis CZ interactions
- 4a: Same-row for ALL L-rows: suffix cascade → adjacent CZ → uncascade → Z on odd columns
- 4b: Cross-row for right-closed pairs $(r, r{+}1)$: prefix cascade → vertical CZ → uncascade → diagonal CZ correction

In [4]:
# =============================================================================
# Construction 2: Ancilla-Free Gamma
#
# CNOT depth: 9L + 12  (exact for L >= 5, verified empirically for L = 5..21)
# Gate count:  O(N)    where N = L^2
# Ancillas:    0
#
# Per-phase CNOT depth breakdown (individual, without inter-phase overlap):
#   Phase 1  (col cascade forward):     L - 1
#   Phase 2a (same-row T, even r >= 2): 2L
#     Primitive 1 on parallel rows. Suffix cascade L-1, CZ layer 2, undo L-1.
#   Phase 2b (skip-row T, 2 rounds):    4L + 7   (for L >= 5)
#     Primitive 3 CNOT depth = 2L + 4 per call (with overlap between undo
#     cascade and correction step, and between prefix cascade and interaction).
#     Adjacent skip-row calls (r=0, r=2) share row r+2 = source row of the
#     next call, requiring 2-round scheduling:
#       Round 1: r = 0, 4, 8, ...  (row triples {0,1,2}, {4,5,6}, ...)
#       Round 2: r = 2, 6, 10, ... (row triples {2,3,4}, {6,7,8}, ...)
#     Within each round, all triples are disjoint → parallel.
#     Two rounds with 1-moment inter-round overlap → 2*(2L+4) - 1 = 4L + 7.
#     NOTE: The doc (Section 5.3) claims Phase 2b depth is 2L + O(1), but
#     this is INCORRECT — it assumes all skip-row calls can be parallelized.
#     They cannot: r=0 (target row 2) and r=2 (source row 2) share row 2.
#   Phase 3  (col cascade inverse):     L - 1
#   Phase 4a (same-row T, all even r):  2L
#   Phase 4b (cross-row T, || pairs):   2L
#     Primitive 2. Prefix cascade L-1, vert CZ 1, undo L-1, vert CZ 1.
#     All right-closed pairs (0,1), (2,3), ... are disjoint → parallel.
#
#   Naive sum: (L-1) + 2L + (4L+7) + (L-1) + 2L + 2L = 12L + 5
#   Inter-phase overlaps save ~3L - 7 moments (cirq greedy scheduling):
#     - Phase 2a (rows >= 4) overlaps with Phase 2b round 1 (rows 0,1,2)
#     - Phase boundary overlaps at each transition
#   ─────────────────────────────────────
#   Total: 12L + 5 - (3L - 7) = 9L + 12
#
# No further depth reduction appears possible:
#   - Phase 2b requires 2-round scheduling (not reducible to 1 round)
#   - Phases 2a/2b share even rows → must be partially sequential
#   - Phases 4a/4b share even rows → must be sequential
#   - Column cascades touch all rows → no overlap with adjacent phases
#   - Inter-phase overlaps are already fully exploited by cirq scheduling
# =============================================================================

# --- Row cascade primitives ---
# Each cascade is depth L-1 (sequential CNOTs along the row).

def suffix_cascade_ops(sq: Dict, r: int, L: int) -> List[cirq.Operation]:
    """Right-to-left CNOT cascade on row r: CNOT(q[c+1] -> q[c]) for c from L-2 down to 0.
    After this, qubit (r,c) holds XOR of s_{r,c'} for c' >= c. CNOT depth: L-1."""
    return [cirq.CNOT(sq[(r, c + 1)], sq[(r, c)]) for c in range(L - 2, -1, -1)]


def undo_suffix_cascade_ops(sq: Dict, r: int, L: int) -> List[cirq.Operation]:
    """Undo suffix cascade: left-to-right CNOT(q[c+1] -> q[c]) for c from 0 to L-2. CNOT depth: L-1."""
    return [cirq.CNOT(sq[(r, c + 1)], sq[(r, c)]) for c in range(L - 1)]


def prefix_cascade_ops(sq: Dict, r: int, L: int) -> List[cirq.Operation]:
    """Left-to-right CNOT cascade on row r: CNOT(q[c-1] -> q[c]) for c from 1 to L-1.
    After this, qubit (r,c) holds XOR of s_{r,c'} for c' <= c. CNOT depth: L-1."""
    return [cirq.CNOT(sq[(r, c - 1)], sq[(r, c)]) for c in range(1, L)]


def undo_prefix_cascade_ops(sq: Dict, r: int, L: int) -> List[cirq.Operation]:
    """Undo prefix cascade: right-to-left CNOT(q[c-1] -> q[c]) for c from L-1 down to 1. CNOT depth: L-1."""
    return [cirq.CNOT(sq[(r, c - 1)], sq[(r, c)]) for c in range(L - 1, 0, -1)]


# --- Primitive 1: Same-row T(x,x) ---

def same_row_T_ops(sq: Dict, r: int, L: int) -> List[cirq.Operation]:
    """Implement (-1)^{T(x,x)} = (-1)^{XOR_{p<c'} x_p * x_{c'}} for row r.
    Circuit: suffix cascade -> adjacent CZ -> undo cascade -> Z on odd columns.
    CNOT depth: 2L (suffix L-1, CZ layer 2, undo L-1; Z corrections are single-qubit, depth 0)."""
    ops = []
    ops.extend(suffix_cascade_ops(sq, r, L))
    for c in range(L - 1):
        ops.append(cirq.CZ(sq[(r, c)], sq[(r, c + 1)]))
    ops.extend(undo_suffix_cascade_ops(sq, r, L))
    for c in range(1, L, 2):  # odd columns — single-qubit Z, no CNOT depth
        ops.append(cirq.Z(sq[(r, c)]))
    return ops


# --- Primitive 2: Cross-row T(x,y) for adjacent rows ---

def cross_row_adjacent_T_ops(sq: Dict, r1: int, r2: int, L: int) -> List[cirq.Operation]:
    """Implement (-1)^{T(x,y)} = (-1)^{XOR_{p<c'} x_p * y_{c'}} where x on row r1, y on adjacent row r2.
    Circuit: prefix cascade on r1 -> vertical CZ -> undo cascade -> vertical CZ correction.
    CNOT depth: 2L (prefix L-1, vert CZ 1, undo L-1, vert CZ 1)."""
    ops = []
    ops.extend(prefix_cascade_ops(sq, r1, L))
    for c in range(L):
        ops.append(cirq.CZ(sq[(r1, c)], sq[(r2, c)]))
    ops.extend(undo_prefix_cascade_ops(sq, r1, L))
    # Diagonal correction: CZ between (r1,c) and (r2,c) for each c
    for c in range(L):
        ops.append(cirq.CZ(sq[(r1, c)], sq[(r2, c)]))
    return ops


# --- Primitive 3: Skip-row T(x,y) for rows 2 apart, routing through intermediate ---

def skip_row_T_ops(sq: Dict, r1: int, r2: int, r_mid: int, L: int) -> List[cirq.Operation]:
    """Implement (-1)^{T(x,y)} where x on row r1, y on row r2=r1+2,
    routing through intermediate row r_mid=r1+1.

    Circuit (4 steps):
      1. Prefix cascade on row r1 (depth L-1)
      2. For each column: CZ-CNOT-CZ-CNOT interaction gadget (depth 4, columns parallel)
      3. Undo prefix cascade on r1 (depth L-1)
      4. For each column: CZ-CNOT-CZ-CNOT correction gadget (depth 4, columns parallel)
         Uses 4-gate gadget (not 3-gate) to avoid contamination by intermediary row bits.

    CNOT depth per call: 2L + 4  (with partial overlaps:
      step 1's last moment overlaps with step 2's first CZ (different rows),
      step 3's undo cascade overlaps partially with step 4's first CZ)
    Touches rows r1, r_mid, r2."""
    ops = []
    # Step 1: Prefix cascade on row r1
    ops.extend(prefix_cascade_ops(sq, r1, L))
    # Step 2: Interaction gadget (all columns parallel, depth 4)
    for c in range(L):
        ops.append(cirq.CZ(sq[(r_mid, c)], sq[(r2, c)]))    # pre-cancel
        ops.append(cirq.CNOT(sq[(r1, c)], sq[(r_mid, c)]))   # copy prefix to intermediary
        ops.append(cirq.CZ(sq[(r_mid, c)], sq[(r2, c)]))     # interaction
        ops.append(cirq.CNOT(sq[(r1, c)], sq[(r_mid, c)]))   # restore intermediary
    # Step 3: Undo prefix cascade on r1
    ops.extend(undo_prefix_cascade_ops(sq, r1, L))
    # Step 4: Correction gadget (all columns parallel, depth 4)
    for c in range(L):
        ops.append(cirq.CZ(sq[(r_mid, c)], sq[(r2, c)]))     # pre-cancel
        ops.append(cirq.CNOT(sq[(r1, c)], sq[(r_mid, c)]))   # copy
        ops.append(cirq.CZ(sq[(r_mid, c)], sq[(r2, c)]))     # interaction
        ops.append(cirq.CNOT(sq[(r1, c)], sq[(r_mid, c)]))   # restore
    return ops


# --- Assemble the full ancilla-free Gamma ---

def build_gamma_ancilla_free(L: int):
    """Build the ancilla-free Gamma circuit.
    Gamma = D_D * C^{-1} * D_B * C
    Returns (circuit, system_qubit_list).

    CNOT depth: 9L + 12  (exact for L >= 5, verified empirically L = 5..21)
    Gate count:  O(N)
    Ancillas:    0"""
    sq = make_system_qubits(L)
    ops = []

    # Phase 1: C (column parity cascade forward) — depth L-1
    ops.extend(column_parity_cascade_ops(sq, L, inverse=False))

    # Phase 2a: D_B same-row terms for L-rows r >= 2 — depth 2L
    # All such rows are independent (disjoint qubits) -> run in parallel
    for r in range(2, L, 2):
        ops.extend(same_row_T_ops(sq, r, L))

    # Phase 2b: D_B skip-row terms — depth 4L+7 (for L >= 5)
    # 2-round scheduling needed: consecutive calls (r=0, r=2) share row 2.
    # Round 1: r = 0, 4, 8, ... (disjoint row triples)
    # Round 2: r = 2, 6, 10, ... (disjoint row triples)
    skip_rows = [r for r in range(0, L, 2) if r + 2 <= L - 1]
    for r in skip_rows[0::2]:   # Round 1
        ops.extend(skip_row_T_ops(sq, r, r + 2, r + 1, L))
    for r in skip_rows[1::2]:   # Round 2
        ops.extend(skip_row_T_ops(sq, r, r + 2, r + 1, L))

    # Phase 3: C^{-1} (column parity cascade inverse) — depth L-1
    ops.extend(column_parity_cascade_ops(sq, L, inverse=True))

    # Phase 4a: D_D same-row terms for ALL L-rows — depth 2L
    # All even rows are independent -> run in parallel
    for r in range(0, L, 2):
        ops.extend(same_row_T_ops(sq, r, L))

    # Phase 4b: D_D cross-row terms for right-closed pairs — depth 2L
    # All pairs (0,1), (2,3), (4,5), ... are disjoint -> run in parallel
    for r in range(0, L - 1, 2):
        ops.extend(cross_row_adjacent_T_ops(sq, r, r + 1, L))

    circuit = cirq.Circuit(ops)
    sys_list = [sq[(r, c)] for r in range(L) for c in range(L)]
    return circuit, sys_list


# Quick check and depth verification
c2, sys2 = build_gamma_ancilla_free(3)
print(f"Construction 2 (L=3): {len(c2)} moments, {len(sys2)} sys qubits (no ancillas)")
for L_test in [5, 7, 9]:
    c_test, _ = build_gamma_ancilla_free(L_test)
    expected = 9 * L_test + 12
    print(f"  L={L_test}: depth = {len(c_test)}, expected 9L+12 = {expected}, match = {len(c_test) == expected}")

Construction 2 (L=3): 31 moments, 9 sys qubits (no ancillas)
  L=5: depth = 57, expected 9L+12 = 57, match = True
  L=7: depth = 75, expected 9L+12 = 75, match = True
  L=9: depth = 93, expected 9L+12 = 93, match = True


## Verification Utilities

Since $\Gamma$ is built entirely from CNOT, CZ, and Z gates (all Clifford), we can simulate it **classically** on any computational basis state: track bits through CNOTs and accumulate phase from CZ and Z gates. This is $O(\text{gates})$ per state and works for arbitrarily many qubits — no quantum simulator needed.

For small $L$ (e.g., $L=3$), we also do an exact full-unitary comparison.

We verify the **property ($\star$)**: for every vertical hop $(r,c) \leftrightarrow (r+1,c)$ and pair of basis states $|s\rangle, |s'\rangle$ differing only at those two sites, $\gamma_s \cdot \gamma_{s'} = (-1)^P$ where $P$ is the parity of all qubits between the two sites in the snake JWT ordering.

In [5]:
# --- Classical Clifford simulation (track bits + phase) ---
# Since Gamma is built from CNOT, CZ, and Z gates (all Clifford),
# we can simulate it classically: track computational basis bits
# through CNOTs and accumulate phase from CZ and Z gates.
# This is O(#gates) per basis state — works for any qubit count.

def classical_sim_phase(ops_list, qubit_to_idx, n_qubits, basis_state_bits):
    """Simulate Clifford circuit on a computational basis state.
    Returns (phase, final_bits) where phase is +1 or -1."""
    bits = list(basis_state_bits)
    phase = 0  # accumulate mod 2
    for op in ops_list:
        gate = op.gate
        qubits = op.qubits
        if isinstance(gate, cirq.ops.common_gates.CNotPowGate) and gate.exponent == 1:
            ctrl_idx = qubit_to_idx[qubits[0]]
            tgt_idx = qubit_to_idx[qubits[1]]
            bits[tgt_idx] ^= bits[ctrl_idx]
        elif isinstance(gate, cirq.ops.common_gates.CZPowGate) and gate.exponent == 1:
            a_idx = qubit_to_idx[qubits[0]]
            b_idx = qubit_to_idx[qubits[1]]
            phase ^= (bits[a_idx] & bits[b_idx])
        elif isinstance(gate, cirq.ops.common_gates.ZPowGate) and gate.exponent == 1:
            idx = qubit_to_idx[qubits[0]]
            phase ^= bits[idx]
        else:
            raise ValueError(f"Unsupported gate: {gate}")
    return (-1)**phase, bits


def get_phase_classical(circuit: cirq.Circuit, qubit_order: List[cirq.Qid],
                        basis_state: int) -> float:
    """Get the diagonal phase for a basis state (ancilla-free circuit)."""
    n = len(qubit_order)
    q2i = {q: i for i, q in enumerate(qubit_order)}
    bits = [(basis_state >> (n - 1 - i)) & 1 for i in range(n)]
    all_ops = [op for moment in circuit for op in moment]
    phase, _ = classical_sim_phase(all_ops, q2i, n, bits)
    return phase


def get_phase_classical_with_ancillas(circuit: cirq.Circuit,
                                       sys_qubits: List[cirq.Qid],
                                       anc_qubits: List[cirq.Qid],
                                       basis_state: int) -> float:
    """Get the diagonal phase for a basis state (ancilla circuit).
    Ancillas start at |0> and must return to |0>."""
    n_sys = len(sys_qubits)
    n_anc = len(anc_qubits)
    all_qubits = sys_qubits + anc_qubits
    q2i = {q: i for i, q in enumerate(all_qubits)}
    bits = [(basis_state >> (n_sys - 1 - i)) & 1 for i in range(n_sys)] + [0] * n_anc
    all_ops = [op for moment in circuit for op in moment]
    phase, final_bits = classical_sim_phase(all_ops, q2i, n_sys + n_anc, bits)
    # Verify ancillas return to |0>
    for i in range(n_anc):
        assert final_bits[n_sys + i] == 0, f"Ancilla {i} not disentangled!"
    return phase


# --- Phase extraction for small L (full unitary, for exact comparison) ---

def extract_diagonal_small(circuit: cirq.Circuit, qubit_order: List[cirq.Qid]) -> np.ndarray:
    """Extract diagonal of the unitary. Returns array of +1/-1 values."""
    U = circuit.unitary(qubit_order=qubit_order)
    return np.real(np.diag(U))


def extract_system_diagonal_with_ancillas(circuit: cirq.Circuit,
                                           sys_qubits: List[cirq.Qid],
                                           anc_qubits: List[cirq.Qid]) -> np.ndarray:
    """For Construction 1: extract system-subspace diagonal.
    Ancillas start and end in |0>. Project onto ancilla=|0> subspace."""
    all_qubits = sys_qubits + anc_qubits
    U = circuit.unitary(qubit_order=all_qubits)
    n_sys = len(sys_qubits)
    n_anc = len(anc_qubits)
    dim_sys = 2 ** n_sys
    dim_anc = 2 ** n_anc
    diag = np.zeros(dim_sys)
    for s in range(dim_sys):
        full_idx = s * dim_anc
        diag[s] = np.real(U[full_idx, full_idx])
    return diag


# --- Property (star) verification ---

def verify_property_star(phase_fn, L: int, num_samples: int = 200,
                         seed: int = 42) -> Tuple[int, int]:
    """Verify property (star) for sampled basis states and all vertical hops.
    phase_fn(basis_state: int) -> float (+1 or -1).
    Returns (num_checked, num_passed)."""
    rng = np.random.default_rng(seed)
    N = L * L
    checked = 0
    passed = 0

    for _ in range(num_samples):
        bits = rng.integers(0, 2, size=N)
        s_idx = sum(int(b) << (N - 1 - i) for i, b in enumerate(bits))

        gamma_s = phase_fn(s_idx)

        for r in range(L - 1):
            for c in range(L):
                i_rc = r * L + c
                i_rc1 = (r + 1) * L + c
                if bits[i_rc] == bits[i_rc1]:
                    continue
                s_prime_idx = s_idx ^ (1 << (N - 1 - i_rc)) ^ (1 << (N - 1 - i_rc1))
                gamma_s_prime = phase_fn(s_prime_idx)

                between = sites_between(r, c, r + 1, L)
                P = 0
                for site_snake in between:
                    sr, sc = snake_to_rc(site_snake, L)
                    raster_idx = sr * L + sc
                    P ^= int(bits[raster_idx])

                expected = (-1) ** P
                actual = gamma_s * gamma_s_prime
                checked += 1
                if abs(actual - expected) < 1e-6:
                    passed += 1

    return checked, passed


print("Verification utilities defined.")

Verification utilities defined.


## Verification: Compare Constructions

In [6]:
def run_small_L_verification(L: int):
    """Full unitary comparison for small L (L=3: 9 sys + 3 anc = 12 qubits)."""
    print(f"\n{'='*60}")
    print(f"Verification for L={L} ({L*L} qubits) — full unitary")
    print(f"{'='*60}")

    # Construction 1
    c1_circuit, c1_sys, c1_anc = build_gamma_with_ancillas(L)
    diag1 = extract_system_diagonal_with_ancillas(c1_circuit, c1_sys, c1_anc)

    # Construction 2
    c2_circuit, c2_sys = build_gamma_ancilla_free(L)
    diag2 = extract_diagonal_small(c2_circuit, c2_sys)

    # Check both are +/-1
    assert np.all(np.abs(np.abs(diag1) - 1.0) < 1e-6), "C1 diagonal not +/-1!"
    assert np.all(np.abs(np.abs(diag2) - 1.0) < 1e-6), "C2 diagonal not +/-1!"

    # Compare
    match = np.allclose(diag1, diag2, atol=1e-6)
    print(f"  Diagonals match: {match}")
    if not match:
        diffs = np.where(np.abs(diag1 - diag2) > 1e-6)[0]
        print(f"  {len(diffs)} mismatches out of {len(diag1)}")
        for idx in diffs[:5]:
            print(f"    state {idx}: C1={diag1[idx]:.1f}, C2={diag2[idx]:.1f}")

    # Property (star) for both
    N = L * L
    n_samples = min(500, 2**N)
    checked1, passed1 = verify_property_star(lambda s: diag1[s], L, num_samples=n_samples)
    checked2, passed2 = verify_property_star(lambda s: diag2[s], L, num_samples=n_samples)
    print(f"  Property (star) C1: {passed1}/{checked1}")
    print(f"  Property (star) C2: {passed2}/{checked2}")

    return match

# L=3: exact unitary comparison (9 sys + 3 anc = 12 qubits)
run_small_L_verification(3)


Verification for L=3 (9 qubits) — full unitary


  Diagonals match: True
  Property (star) C1: 1486/1486
  Property (star) C2: 1486/1486


True

In [7]:
def run_sampled_verification(L: int, num_samples: int = 1000, seed: int = 42):
    """Sample-based comparison using classical Clifford simulation.
    Works for any L since we just track bits + phase through gates."""
    print(f"\n{'='*60}")
    print(f"Verification for L={L} ({L*L} qubits) — {num_samples} samples (classical sim)")
    print(f"{'='*60}")

    N = L * L
    rng = np.random.default_rng(seed)

    # Build circuits
    c1_circuit, c1_sys, c1_anc = build_gamma_with_ancillas(L)
    c2_circuit, c2_sys = build_gamma_ancilla_free(L)

    # Sample random basis states
    samples = set()
    while len(samples) < num_samples:
        samples.add(int(rng.integers(0, 2**N)))
    samples = sorted(samples)

    mismatches = 0
    for i, s in enumerate(samples):
        if (i + 1) % 200 == 0:
            print(f"  Progress: {i+1}/{num_samples}")
        p1 = get_phase_classical_with_ancillas(c1_circuit, c1_sys, c1_anc, s)
        p2 = get_phase_classical(c2_circuit, c2_sys, s)
        if abs(p1 - p2) > 1e-6:
            mismatches += 1
            if mismatches <= 3:
                print(f"  MISMATCH at state {s}: C1={p1}, C2={p2}")

    match = mismatches == 0
    print(f"  Diag match: {num_samples - mismatches}/{num_samples}")

    # Property (star) for C2
    phase_fn = lambda s: get_phase_classical(c2_circuit, c2_sys, s)
    checked, passed = verify_property_star(phase_fn, L, num_samples=min(200, num_samples))
    print(f"  Property (star) C2: {passed}/{checked}")

    return match

In [8]:
# L=4: 16 sys + 4 anc = 20 qubits
run_sampled_verification(L=4, num_samples=500)


Verification for L=4 (16 qubits) — 500 samples (classical sim)
  Progress: 200/500
  Progress: 400/500
  Diag match: 500/500
  Property (star) C2: 1165/1165


True

In [9]:
# L=5: 25 sys + 5 anc = 30 qubits
run_sampled_verification(L=5, num_samples=1000)


Verification for L=5 (25 qubits) — 1000 samples (classical sim)
  Progress: 200/1000
  Progress: 400/1000


  Progress: 600/1000
  Progress: 800/1000
  Progress: 1000/1000
  Diag match: 1000/1000
  Property (star) C2: 2007/2007


True

In [10]:
# L=7: 49 sys + 7 anc = 56 qubits — classical sim handles this easily
run_sampled_verification(L=7, num_samples=1000)


Verification for L=7 (49 qubits) — 1000 samples (classical sim)
  Progress: 200/1000
  Progress: 400/1000
  Progress: 600/1000
  Progress: 800/1000


  Progress: 1000/1000
  Diag match: 1000/1000


  Property (star) C2: 4224/4224


True

## Depth Analysis Summary

| Construction | Depth Formula | Ancillas | Notes |
|---|---|---|---|
| Ancilla-based (Jiang et al.) | $7L - 3$ | $L$ | Exact for all $L$ |
| Ancilla-free (2-round optimized) | $9L + 12$ | $0$ | Exact for $L \ge 5$ |

**Key optimization**: Phase 2b (skip-row $T$) uses 2-round scheduling to keep depth $O(L)$.
Without it, consecutive skip-row calls share rows and depth degrades to $O(L^2)$.

**Primitive depths** (single invocation):
- Same-row $T(x,x)$: $2L+1$ (suffix cascade $L{-}1$, CZ layer depth 2, undo $L{-}1$, Z depth 1)
- Cross-row $T(x,y)$: $2L$ (prefix cascade $L{-}1$, vert CZ 1, undo $L{-}1$, vert CZ 1)
- Skip-row $T(x,y)$: $2L+4$ (prefix $L{-}1$, 4-gate gadget depth 4, undo $L{-}1$, 4-gate correction depth 4, minus overlap)

**No further depth reduction possible**: Phases 2a/2b cannot run in parallel (share even rows). Phases 4a/4b cannot run in parallel (share even rows). Same-row $T$ uses suffix cascades while cross-row $T$ uses prefix cascades — they cannot be fused.

In [11]:
# Verify depth formulas across multiple L values
print(f"{'L':>3} | {'C1 (ancilla)':>14} | {'C2 (ancilla-free)':>18} | {'7L-3':>5} | {'9L+12':>5}")
print("-" * 70)
for L in [3, 5, 7, 9, 11]:
    c1_circ, _, _ = build_gamma_with_ancillas(L)
    c2_circ, _ = build_gamma_ancilla_free(L)
    d1, d2 = len(c1_circ), len(c2_circ)
    g1 = sum(len(m.operations) for m in c1_circ)
    g2 = sum(len(m.operations) for m in c2_circ)
    print(f"{L:3d} | {d1:4d} depth {g1:4d}g | {d2:4d} depth {g2:5d}g | {7*L-3:5d} | {9*L+12:5d}")

  L |   C1 (ancilla) |  C2 (ancilla-free) |  7L-3 | 9L+12
----------------------------------------------------------------------
  3 |   18 depth   47g |   31 depth    71g |    18 |    39
  5 |   32 depth  139g |   57 depth   242g |    32 |    57
  7 |   46 depth  279g |   75 depth   513g |    46 |    75
  9 |   60 depth  467g |   93 depth   884g |    60 |    93
 11 |   74 depth  703g |  111 depth  1355g |    74 |   111
